# 📄 Resume RAG System for Intelligent Job Matching

This project implements a **Retrieval-Augmented Generation (RAG)** system for semantic resume retrieval and job matching.

## Project Workflow

01. Resume PDFs
02. PDF Text Extraction
03. Intelligent Chunking
04. Metadata Extraction
05. Embedding Generation
06. Chroma Vector Database
07. Semantic Retrieval
08. Hybrid Search (BM25)
09. Candidate Ranking
10. JSON Output 

This project uses the following libraries:

| Library | Purpose |
|----------|----------|
| PyMuPDF | Read PDF resumes |
| sentence-transformers | Generate embeddings |
| ChromaDB | Vector database |
| LangChain | Text splitting |
| Rank-BM25 | Keyword search |
| spaCy | Metadata extraction |
| pandas | Data analysis |
| NumPy | Numerical operations |
| tqdm | Progress bars |

These libraries together implement the complete Retrieval-Augmented Generation (RAG) pipeline.

In [19]:
# Uncomment and run only once

!pip install -r ../requirements.txt

  Using cached langchain_openrouter-0.2.7-py3-none-any.whl.metadata (3.4 kB)
  Using cached openrouter-0.11.46-py3-none-any.whl.metadata (9.4 kB)
  Using cached jsonpath_python-1.1.6-py3-none-any.whl.metadata (17 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached pydantic_core-2.41.5-cp314-cp314-win_amd64.whl.metadata (7.4 kB)
Using cached langchain_openrouter-0.2.7-py3-none-any.whl (29 kB)
Using cached openrouter-0.11.46-py3-none-any.whl (837 kB)
Using cached pydantic-2.12.5-py3-none-any.whl (463 kB)
Using cached pydantic_core-2.41.5-cp314-cp314-win_amd64.whl (2.0 MB)
Using cached jsonpath_python-1.1.6-py3-none-any.whl (14 kB)

  Attempting uninstall: pydantic-core

    Found existing installation: pydantic_core 2.46.4

    Uninstalling pydantic_core-2.46.4:

      Successfully uninstalled pydantic_core-2.46.4

   ---------------------------------------- 0/5 [pydantic-core]
   ---------------------------------------- 0/5 [pydantic-core]
   ------------

  You can safely remove it manually.

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# Download the spaCy English model (run once)
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ------ --------------------------------- 2.1/12.8 MB 27.3 MB/s eta 0:00:01
     --------------------------- ------------ 8.7/12.8 MB 31.0 MB/s eta 0:00:01
     --------------------------------------  12.6/12.8 MB 27.9 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 25.4 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Import Required Libraries

In this section we import all libraries required for the project.

The imports are grouped according to their purpose:

- File Handling
- PDF Processing
- Machine Learning
- Embedding Models
- Vector Database
- Metadata Extraction
- Visualization

In [20]:
import os
import json
import warnings
from pathlib import Path

import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

# LangChain
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# OpenRouter
from langchain_openrouter import ChatOpenRouter

# ChromaDB
from langchain_chroma import Chroma

# Hybrid Search
from rank_bm25 import BM25Okapi

print("✅ Libraries imported successfully.")

✅ Libraries imported successfully.


# Configure the Environment

This project uses **OpenRouter** as the gateway for accessing Large Language Models (LLMs).

The API key is stored securely in a `.env` file located in the project root.

Example:

```text
OPENROUTER_API_KEY=your_openrouter_api_key
```

Using environment variables keeps sensitive credentials out of the source code and makes the project easier to deploy and share.

In [13]:
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv("../.env")

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("❌ OPENROUTER_API_KEY not found in .env file.")

print("✅ OpenRouter API Key loaded successfully.")

✅ OpenRouter API Key loaded successfully.


# Initialize AI Models

A Retrieval-Augmented Generation (RAG) system consists of two primary AI components:

### 1. Embedding Model

The embedding model converts textual data into dense numerical vectors that capture semantic meaning. These embeddings enable efficient similarity search within the vector database.

Embedding Model:
- **text-embedding-3-small**

### 2. Large Language Model (LLM)

The Large Language Model (LLM) is responsible for understanding natural language and generating human-readable responses.

In this project, the LLM is used to:

- Extract resume metadata
- Explain candidate-job matches
- Generate structured JSON responses
- Support future enhancements such as resume summarization

LLM:
- **OpenAI GPT-4o Mini** (accessed through OpenRouter)

Separating retrieval (Embeddings) from reasoning (LLM) is a common architecture used in modern RAG systems.

In [25]:
# ============================================================
# Initialize Embedding Model
# ============================================================

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embedding model initialized : sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Embedding model initialized : sentence-transformers/all-MiniLM-L6-v2


In [26]:
# ============================================================
# Initialize LLM
# ============================================================

llm = ChatOpenRouter(
    model="openai/gpt-4o-mini",
    temperature=0.2,
    api_key=OPENROUTER_API_KEY,
    max_tokens=150
)

print("✅ LLM initialized : openai/gpt-4o-mini")

✅ LLM initialized : openai/gpt-4o-mini


# Load Resume Documents

The first stage of a Retrieval-Augmented Generation (RAG) pipeline is **document ingestion**.

In this step, we load all resume PDF files from the `data/resumes/` directory using LangChain's **PyPDFLoader**.

Each PDF is converted into one or more **Document** objects.

A `Document` contains:

- **page_content** – The extracted text from each page.
- **metadata** – Information such as the source file path and page number.

These documents will later be:
- Chunked into smaller sections
- Converted into embeddings
- Stored in the vector database for semantic retrieval

In [27]:
# ============================================================
# Configure Project Directories
# ============================================================

from pathlib import Path

# Project Root
PROJECT_ROOT = Path.cwd().parent

# Data Directories
DATA_DIR = PROJECT_ROOT / "data"

RESUME_DIR = DATA_DIR / "resumes"

JOB_DESCRIPTION_DIR = DATA_DIR / "job_descriptions"

OUTPUT_DIR = PROJECT_ROOT / "outputs"

VECTOR_STORE_DIR = PROJECT_ROOT / os.getenv(
    "VECTOR_STORE_DIR",
    "data/chroma_db"
)

# Create directories if they don't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root      : {PROJECT_ROOT}")
print(f"Resume Directory  : {RESUME_DIR}")
print(f"Vector Store      : {VECTOR_STORE_DIR}")

Project Root      : d:\GitHub\RAG-Based-Profile-matching
Resume Directory  : d:\GitHub\RAG-Based-Profile-matching\data\resumes
Vector Store      : d:\GitHub\RAG-Based-Profile-matching\chroma_db


In [29]:
# ============================================================
# Find Resume PDFs
# ============================================================

resume_files = sorted(RESUME_DIR.glob("*.pdf"))

print(f"Found {len(resume_files)} resume(s).\n")

for pdf in resume_files:
    print(pdf.name)

Found 2 resume(s).

alex_green.pdf
alice_smith.pdf
